In [1]:
from collections import deque
import time


## 1. Enter the maze

Use `S` for the start, `E` for the destination, `#` for a wall, and `.` for an open cell. Enter rows without spaces. The maze must have exactly one start and one destination.

In [2]:
while True:
    try:
        rows = int(input("Number of rows: "))
        cols = int(input("Number of columns: "))
        if rows > 0 and cols > 0:
            break
        print("Enter positive numbers.")

print("Use S = start, E = end, # = wall, * = open cell")

while True:
    maze = []
    for r in range(rows):
        while True:
            row = input(f"Row {r + 1}: ").strip()
            if len(row) != cols:
                print(f"Enter exactly {cols} characters.")
            elif any(ch not in "SE#*" for ch in row):
                print("Only S, E, # and . are allowed.")
            else:
                maze.append(row)
                break

    if sum(row.count("S") for row in maze) == 1 and sum(row.count("E") for row in maze) == 1:
        break
    print("Enter the maze again with exactly one S and one E.")

for r in range(rows):
    for c in range(cols):
        if maze[r][c] == "S":
            start = (r, c)
        elif maze[r][c] == "E":
            end = (r, c)

print("Start:", start, "End:", end)


Number of rows:  4
Number of columns:  4


Use S = start, E = end, # = wall, . = open cell


Row 1:  S . # #


Enter exactly 4 characters.


Row 1:  # . # #


Enter exactly 4 characters.


... (4 outputs left out)

Row 1:  S . # #


Enter exactly 4 characters.


Row 1:  S.##
Row 2:  #.##
Row 3:  #.##
Row 4:  #..E


Start: (0, 0) End: (3, 3)


## 2. Find valid neighbours

In [8]:
def get_neighbours(position):
    r, c = position
    neighbours = []
    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < rows and 0 <= nc < cols:
            if maze[nr][nc] != "#":
                neighbours.append((nr, nc))
    return neighbours


## 3. Breadth First Search (BFS)

BFS uses a queue and explores level by level. It finds a shortest path when all moves have equal cost.

In [9]:
def bfs(start, end):
    queue = deque([[start]])
    visited = {start}
    explored = 0

    while queue:
        path = queue.popleft()
        node = path[-1]
        explored += 1

        if node == end:
            return path, explored

        for neighbour in get_neighbours(node):
            if neighbour not in visited:
                visited.add(neighbour)
                queue.append(path + [neighbour])

    return None, explored


## 4. Depth First Search (DFS)

DFS uses a stack and explores one branch deeply before trying other branches. It does not guarantee a shortest path. With this stack, the last neighbour added is explored first.

In [10]:
def dfs(start, end):
    stack = [[start]]
    visited = {start}
    explored = 0

    while stack:
        path = stack.pop()
        node = path[-1]
        explored += 1

        if node == end:
            return path, explored

        for neighbour in get_neighbours(node):
            if neighbour not in visited:
                visited.add(neighbour)
                stack.append(path + [neighbour])

    return None, explored


## 5. Compare performance

Path length counts moves, so it is the number of cells in the path minus one. Nodes explored counts cells removed from the queue or stack, including the destination. Time measures only the search call. A single timing on a small maze can vary between runs.

In [11]:
t1 = time.perf_counter()
bfs_path, bfs_nodes = bfs(start, end)
t2 = time.perf_counter()
bfs_time = (t2 - t1) * 1000

t1 = time.perf_counter()
dfs_path, dfs_nodes = dfs(start, end)
t2 = time.perf_counter()
dfs_time = (t2 - t1) * 1000

if bfs_path is not None:
    bfs_moves = len(bfs_path) - 1
else:
    bfs_moves = "No path"

if dfs_path is not None:
    dfs_moves = len(dfs_path) - 1
else:
    dfs_moves = "No path"

print("Algorithm | Moves | Explored | Time (ms)")
print("BFS |", bfs_moves, "|", bfs_nodes, "|", round(bfs_time, 6))
print("DFS |", dfs_moves, "|", dfs_nodes, "|", round(dfs_time, 6))

Algorithm   Moves       Explored    Time (ms)   
BFS         6           7           0.284200    
DFS         6           7           0.150500    


## 6. Display the paths

`*` marks the route. Start and destination symbols are preserved.

In [13]:
def show_path(path):
    if path is None:
        print("No path exists.")
        return

    grid = [list(row) for row in maze]

    for r, c in path:
        if grid[r][c] == ".":
            grid[r][c] = "*"

    print("Moves:", len(path) - 1)
    for row in grid:
        print(" ".join(row))


print("BFS PATH")
show_path(bfs_path)

print("\nDFS PATH")
show_path(dfs_path)

print("\nS = Start | E = End | # = Wall | * = Path")

BFS PATH
Moves: 6
S * # #
# * # #
# * # #
# * * E

DFS PATH
Moves: 6
S * # #
# * # #
# * # #
# * * E

S = Start | E = End | # = Wall | * = Path
